## RAG PIPELINEs -Data ingestion to Vector DB pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path
from langchain_core.documents import Document


/home/nabin/www/rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# from pathlib import Path
# from langchain_core.documents import Document

# all_documents = []

# for md_file in Path("../data").glob("**/*.md"):
#     text = md_file.read_text(encoding="utf-8")

#     all_documents.append(
#         Document(
#             page_content=text,
#             metadata={
#                 "source_file": md_file.name,
#                 "file_type": "md",
#             },
#         )
#     )

# print(all_documents)

In [3]:
### Read all the pdf's inside the directory
from langchain_community.document_loaders import UnstructuredMarkdownLoader


def process_all_md_files(md_directory):
    """Process all MD files in a directory"""
    all_documents = []
    md_dir = Path(md_directory)
    
    # Find all md files recursively
    md_files = list(md_dir.glob("**/*.md"))
    
    print(f"Found {len(md_files)} MD files to process")
    
    for md_file in md_files:
        print(f"\nProcessing: {md_file.name}")
        try:
            text = md_file.read_text(encoding="utf-8")
            all_documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source_file": md_file.name,
                        "file_type": "md",
                    },
                )
            )
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all MD files in the data directory
all_md_documents = process_all_md_files("../data")

Found 54 MD files to process

Processing: robotBdd.md

Processing: scenarioOutline.md

Processing: test_cafe_blog.md

Processing: organizeYourCompanyTheGeekWay.md

Processing: nightwatchSetup.md

Processing: tablenodes.md

Processing: codeceptJs.md

Processing: dockerSelenium.md

Processing: jest-puppeteer.md

Processing: TDDInGo.md

Processing: tusProtocol.md

Processing: understandingHttpAndSocket.md

Processing: debugging.md

Processing: BDDWithFlutter.md

Processing: k6-01.md

Processing: k6-02.md

Processing: htmx.md

Processing: contract-testing-with-pact-(javascript).md

Processing: 20-beginner-tech-terms.md

Processing: browser-automation-protocol.md

Processing: are_you_safe_enough_to_take_risks.md

Processing: options-vs-composition-api.md

Processing: ocisWrapper.md

Processing: BDDInGo_1_DE.md

Processing: BDDInGo_2_DE.md

Processing: BDDInGo.md

Processing: cypress_blog_5.md

Processing: cypress_blog_6.md

Processing: cypress_blog_3.md

Processing: cypress_blog_4.md

Proce

In [4]:
all_md_documents

[Document(metadata={'source_file': 'robotBdd.md', 'file_type': 'md'}, page_content='---\ntitle: Setup and run tests with robot framework\nauthorName: Jasmine Baral\nauthorAvatar: https://avatars.githubusercontent.com/u/40829116?v=4\nauthorLink: https://github.com/jasson99/\ncreatedAt: April 6, 2020\ntags: testing, bdd, robotframework, python\nbanner: https://raw.githubusercontent.com/JankariTech/blog/master/src/imgs/fallback_banner.png\n---\n## Robot Framework\n\nThe Robot framework is a generic open-source test automation framework for acceptance testing. It is a keyword-driven testing framework that uses tabular test data syntax. It has easy syntax, using human-readable keywords. This framework is independent of the operating system and application. The core framework is implemented using Python and also runs on Jython(JVM) and IronPython(.NET). This framework provides support for external libraries, tools which are open source and can be used for test automation.\n\nThe test data is

In [5]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [6]:
chunks=split_documents(all_md_documents)
chunks

Split 54 documents into 719 chunks

Example chunk:
Content: ---
title: Setup and run tests with robot framework
authorName: Jasmine Baral
authorAvatar: https://avatars.githubusercontent.com/u/40829116?v=4
authorLink: https://github.com/jasson99/
createdAt: Apr...
Metadata: {'source_file': 'robotBdd.md', 'file_type': 'md'}


[Document(metadata={'source_file': 'robotBdd.md', 'file_type': 'md'}, page_content='---\ntitle: Setup and run tests with robot framework\nauthorName: Jasmine Baral\nauthorAvatar: https://avatars.githubusercontent.com/u/40829116?v=4\nauthorLink: https://github.com/jasson99/\ncreatedAt: April 6, 2020\ntags: testing, bdd, robotframework, python\nbanner: https://raw.githubusercontent.com/JankariTech/blog/master/src/imgs/fallback_banner.png\n---\n## Robot Framework\n\nThe Robot framework is a generic open-source test automation framework for acceptance testing. It is a keyword-driven testing framework that uses tabular test data syntax. It has easy syntax, using human-readable keywords. This framework is independent of the operating system and application. The core framework is implemented using Python and also runs on Jython(JVM) and IronPython(.NET). This framework provides support for external libraries, tools which are open source and can be used for test automation.'),
 Document(metada

## Next part need to add on requirement file

### Embedding And VectorStoreDB

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9106.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### Vector store using chromadb

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: pdf_documents
Existing documents in collection: 5439


In [10]:
### here we have chunks
chunks

[Document(metadata={'source_file': 'robotBdd.md', 'file_type': 'md'}, page_content='---\ntitle: Setup and run tests with robot framework\nauthorName: Jasmine Baral\nauthorAvatar: https://avatars.githubusercontent.com/u/40829116?v=4\nauthorLink: https://github.com/jasson99/\ncreatedAt: April 6, 2020\ntags: testing, bdd, robotframework, python\nbanner: https://raw.githubusercontent.com/JankariTech/blog/master/src/imgs/fallback_banner.png\n---\n## Robot Framework\n\nThe Robot framework is a generic open-source test automation framework for acceptance testing. It is a keyword-driven testing framework that uses tabular test data syntax. It has easy syntax, using human-readable keywords. This framework is independent of the operating system and application. The core framework is implemented using Python and also runs on Jython(JVM) and IronPython(.NET). This framework provides support for external libraries, tools which are open source and can be used for test automation.'),
 Document(metada

In [11]:
### convert the text to embeddings
texts = [doc.page_content for doc in chunks]
# print(texts) # for output 

## Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)
# print(embeddings)


## Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 719 texts...


Batches: 100%|██████████| 23/23 [00:13<00:00,  1.65it/s]


Generated embeddings with shape: (719, 384)
Adding 719 documents to vector store...
Successfully added 719 documents to vector store
Total documents in collection: 6158


## Retriever Pipeline From VectorStore

In [12]:
class RAGRetriever:
    """"Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)


In [13]:
rag_retriever

In [14]:
# rag_retriever.retrieve("What is attention is all you need")
# rag_retriever.retrieve("who write Setup and run tests with robot framework?")
rag_retriever.retrieve("Who wrote github actions?")

Retrieving documents for query: 'Who wrote github actions?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 81.18it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_4e2924a8_697',
  'content': "With a practical example, we've seen how easy it is to set up a basic workflow.\n\nIn summary, GitHub Actions is a powerful tool for speedy development processes, making it easier to deliver high-quality software efficiently.",
  'metadata': {'file_type': 'md',
   'content_length': 223,
   'source_file': 'githubActionBeginner.md',
   'doc_index': 697},
  'similarity_score': 0.29539215564727783,
  'distance': 0.7046078443527222,
  'rank': 1},
 {'id': 'doc_fc9bd283_697',
  'content': "With a practical example, we've seen how easy it is to set up a basic workflow.\n\nIn summary, GitHub Actions is a powerful tool for speedy development processes, making it easier to deliver high-quality software efficiently.",
  'metadata': {'file_type': 'md',
   'doc_index': 697,
   'content_length': 223,
   'source_file': 'githubActionBeginner.md'},
  'similarity_score': 0.29539215564727783,
  'distance': 0.7046078443527222,
  'rank': 2},
 {'id': 'doc_ccc9a6a8_69

### Integration vectordb Context pipeline With LLM output

In [15]:
### simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

# llm=ChatGroq(groq_api_key=groq_api_key,model_name="gemma2-9b-it",temperature=0.1,max_tokens=1024)
# llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)
llm=ChatGroq(groq_api_key=groq_api_key,model_name="meta-llama/llama-4-scout-17b-16e-instruct",temperature=0.1,max_tokens=1024)



## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content


In [19]:
answer=rag_simple("Who wrote CodeceptJs - Setup & BDD?",rag_retriever,llm)
# answer=rag_simple("What topic is written by Nabin Ale?",rag_retriever,llm)
print(answer)

Retrieving documents for query: 'Who wrote CodeceptJs - Setup & BDD?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 18.61it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Hari Bhandari.
